In [11]:
from pathlib import Path
import pandas as pd

DATA = Path(r"C:\Users\Harman\Lumen-IDS\data\MachineLearningCSV\MachineLearningCVE")

df = pd.read_csv(DATA / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
df.columns = df.columns.str.strip()
df.shape

(286467, 79)

In [12]:
df.head()
print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [13]:
df["Label"].value_counts()
df["Label"].value_counts(normalize=True).round(3)

Label
PortScan    0.555
BENIGN      0.445
Name: proportion, dtype: float64

In [14]:
import numpy as np

print("Missing values:", df.isna().sum().sum())
print("Infinite values:", np.isinf(df.select_dtypes("number")).sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 15
Infinite values: 727
Duplicate rows: 72353


In [15]:
cols = ["Flow Duration", "Total Fwd Packets", "Total Backward Packets",
        "Flow Packets/s", "SYN Flag Count"]
df.replace([np.inf, -np.inf], np.nan).groupby("Label")[cols].median()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Flow Packets/s,SYN Flag Count
Label,,,,,
BENIGN,30964.0,2.0,2.0,112.633796,0.0
PortScan,47.0,1.0,1.0,42553.191490,0.0


# First Look at CIC-IDS2017

## 1. Dataset size
- The dataset contains 286,467 rows and 79 columns.
- Each row represents a network flow, and the columns contain traffic measurements and a label.

## 2. Attack distribution
- PortScan traffic accounts for approximately 55.5% of flows.
- Benign traffic accounts for approximately 44.5% of flows.
- The dataset is moderately imbalanced, with more PortScan-labelled flows than benign flows.

## 3. Feature observations
- The median Flow Duration is much lower for PortScan than for benign traffic.
- PortScan has fewer forward and backward packets at the median.
- The median Flow Packets/s is much higher for PortScan.
- The median SYN Flag Count is zero for both classes, so it does not clearly distinguish the classes using the median alone.

These differences suggest that flow duration, packet counts and packet rate may be useful features for distinguishing port scans from normal traffic.

## 4. Data quality
- Missing values: 15.
- Infinite values: 727.
- Duplicate rows: 72,353.

These data-quality issues should be investigated and handled carefully before training machine-learning models.